# Prefilling River Basin Districts and Competent Authorities data

This notebook builds a prefilled version of the data to be submitted for the 4th River Basin Management Plans, under the *River Basin Districts and Competent Authorities* dataflow.

All the SQL lives in the precompiled DuckDB catalog `wise_rbdsuca.duckdb`, committed in this repository:

| schema | content |
| --- | --- |
| `prefill` | views resolving the DiscoData / data lake sources for one country |
| `reference` | views over the reference datasets used by the reference QCs |
| `qc` | the quality checks, used by the *check* notebook |
| `meta` | the `stable_record_id` macro, the dataset inventory and the export scripts |

The result is a SQLite file (descriptive data) and an OGC GeoPackage (spatial data) conforming to the data model of the 4th reporting cycle, documented in the [WFD reporting documentation](https://eeadata.github.io/WISE.WFD.Documentation/TestingPhase/WFDRiverBasinDistrictsAndCompetentAuthorities.html).

> `wise_rbdsuca.duckdb` ships precompiled - no build step needed. If you edit anything under `sql/`, run `python build_catalog.py` from the repository root to refresh it.

In [1]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Initial setup

Opens the catalog and loads the DuckDB extensions the views depend on (`spatial`, `httpfs`, `azure`, `sqlite`).

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wise_local_qc.py").exists())
sys.path.insert(0, str(ROOT))

import geopandas as gpd
import ipywidgets as widgets
from ipyleaflet import GeoData, LayersControl, Map, basemaps

import wise_local_qc as wq

# avoid a locked-file error if this cell is re-run without restarting the kernel
if "con" in globals():
    try:
        globals()["con"].close()
    except Exception:
        pass

con = wq.connect(ROOT / "wise_rbdsuca.duckdb")
wq.current_parameters(con)

{'country_code': 'AT', 'cycle_year': '2022', 'reference_cycle_year': '2022'}

## 2. Available tables

Everything the catalog can resolve. Nothing is downloaded yet: these are views.

In [14]:
wq.list_datasets(con)

,schema_name,table_name,kind,description
0,prefill,CompetentAuthority,descriptive,Prefilled descriptive data for the selected co...
1,prefill,RiverBasinDistrictCompetentAuthority,descriptive,Prefilled descriptive data for the selected co...
2,reference,Country,reference,Reference dataset from the reference reporting...
3,reference,RiverBasinDistrictWFD,reference,Reference dataset from the reference reporting...
4,prefill,SPATIAL_RiverBasinDistrict,spatial,Prefilled spatial data for the selected countr...


## 3. Country selection

Pick the country and the reporting cycle to prefill, then run the next cell to apply the selection. The selection is pushed to the catalog as DuckDB session variables, which every view reads through `getvariable()`.

In [24]:
countries_selection = widgets.Dropdown(
    options=wq.COUNTRIES,  # type: ignore[attr-defined]
    value="AT",
    description="Country:",
)
cycle_selection = widgets.Dropdown(options=["2022", "2016", "2010"], value="2022", description="Cycle:")
widgets.VBox([countries_selection, cycle_selection])

In [32]:
wq.set_parameters(con, country_code=countries_selection.value, cycle_year=cycle_selection.value)
wq.current_parameters(con) 

{'country_code': 'SE', 'cycle_year': '2022', 'reference_cycle_year': '2022'}

## 4. Query the data available for the selected country

Each cell below triggers the download of only what it needs.

In [33]:
wq.preview(con, "prefill.CompetentAuthority")

,euCACode,competentAuthorityName,competentAuthorityNameNL,competentAuthorityNameNLLanguage,acronym,street,city,country,postcode,url,record_id
0,SEBOV,"National Board of Housing, Building and Planning",Boverket,swe,None,Drottninggatan 18,Karlskrona,Sverige,371 23,http://www.boverket.se/,c3090ba6-413e-8509-7086-5e43615b62b6
1,SEBD25,County administrative board of Norrbotten county,Länsstyrelsen Norrbotten,swe,None,Stationsgatan 5,Luleå,Sverige,971 86,http://www.lansstyrelsen.se/Norrbotten,ce723e8b-87fa-7757-edc9-259dbcd1ecbc
2,SEAC24,County administrative board of Västerbotten co...,Länsstyrelsen Västerbotten,swe,None,Storgatan 71 B,Umeå,Sverige,901 86,http://www.lansstyrelsen.se/vasterbotten,098c2b5c-4437-5d21-1498-a10137ecf879
3,SEAB01,County administrative board of Stockholm,Länsstyrelsen i Stockholm,swe,None,Regeringsgatan 66,Stockholm,Sverige,104 22,http://www.lansstyrelsen.se/stockholm,cdcdd3b8-efbd-ed50-0de2-e62733b4b690
4,SE5,5. Skagerrak and Kattegat (Sweden),Vattenmyndigheten för Västerhavets vattendistrikt,swe,Skagerrak and Kattegat,"Länsstyrelsen i Västra Götaland, Ekelundsgatan 1",GÖTEBORG,Sverige,SE-403 40,www.vattenmyndigheterna.se,69797216-8e67-cb12-346b-04821b2b866d
5,SE4K,Municipalities in South Baltic Sea (Sweden) (91),Kommuner i Södra Östersjön (91 st),swe,None,None,None,Sverige,None,Not available,6e2b30fc-d584-45a8-7b9b-eb8268b6a3fb
6,SE4,4. South Baltic Sea (Sweden),Vattenmyndigheten för Södra Östersjöns vattend...,swe,South Baltic Sea,"Länsstyrelsen i Kalmar län, Regeringsgatan 1",KALMAR,Sverige,SE-391 86,www.vattenmyndigheterna.se,deae1db8-dc41-ff3f-e536-6eb960f02f5d
7,SE3K,Municipalities in North Baltic Sea (Sweden) (76),Kommuner i Norra Östersjön (76 st),swe,None,None,None,Sverige,None,Not available,366e3929-f9a2-ca62-50c8-14bc0f5960da
8,SE3,3. North Baltic Sea (Sweden),Vattenmyndigheten för Norra Östersjöns vattend...,swe,North Baltic Sea,"Länsstyrelsen i Västmanland, Västra Ringvägen 1",VÄSTERÅS,Sverige,SE-721 86,www.vattenmyndigheterna.se,b24e4508-1d67-524c-4d95-5a197ec95581
9,SE2,2. Bothnian Sea (Sweden),Vattenmyndigheten för Bottenhavets vattendistrikt,swe,Bothnian Sea,Länsstyrelsen i Västernorrland,HÄRNÖSAND,Sverige,SE-871 86,www.vattenmyndigheterna.se,e7baecd8-26a1-dee1-781f-c728e735c8d0


In [34]:
wq.preview(con, "prefill.RiverBasinDistrictCompetentAuthority")

,euRBDCode,euCACode,mainRole,record_id
0,SE1_INT,SE1,preparationOfRBMP,f4ad27f9-865f-e8d2-2a86-e7a7cad0e01f
1,SE1_INT,SE1,coordinationOfImplementation,5ab36679-1c76-c7fc-b980-9eaddc69dd7f
2,SE2_INT,SE2,economicAnalysis,468acdf9-63a9-4913-c579-d5cb79e5dc82
3,SE3,SE3,economicAnalysis,3188b54c-c1af-3085-b35b-ad7e2138f4d8
4,SE2_INT,SE2,preparationOfProgrammeOfMeasures,4b238da1-fb02-fe22-48e2-39309884b5f1
5,SE3,SE3,preparationOfProgrammeOfMeasures,4c4242a3-f1ad-4659-a6d4-d1e38761a5af
6,SE4,SE4,coordinationOfImplementation,1894f9cb-cd9e-e487-241c-ca4f83e75195
7,SE4,SE4,preparationOfRBMP,fdd8f6f0-9538-8979-e137-e281d86d9ec5
8,SE2_INT,SEAC24,assessmentOfStatusOfGroundwater,9c1bf5d9-4f44-fe98-ccb1-972159cc3b2f
9,SE1_INT,SEBD25,publicParticipation,6b9a5166-1468-3058-d0ba-f0273b218286


In [35]:
con.sql("SELECT * EXCLUDE (geometry_polygon) FROM prefill.SPATIAL_RiverBasinDistrict").df()

,inspireIdLocalId,inspireIdNamespace,inspireIdVersionId,thematicIdIdentifier,thematicIdIdentifierScheme,beginLifespanVersion,endLifespanVersion,predecessorsIdentifier,predecessorsIdentifierScheme,successorsIdentifier,...,nameLanguage,designationPeriodBegin,designationPeriodEnd,zoneType,specialisedZoneType,legalBasisName,legalBasisLink,legalBasisLevel,link,record_id
0,SE1_INT,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE1_INT,euRBDCode,2021-12-02,None,"SE1,SE1103,SE1104,SE1TO",euRBDCode,None,...,swe,2022-09-05,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,c77ee5cd-2e30-992f-c366-8b301fe6228a
1,SE2_INT,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE2_INT,euRBDCode,2021-11-30,None,"SE1102,SE2",euRBDCode,None,...,swe,2022-09-05,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,a2f4384e-6aba-686d-6aa2-6d77f1d52370
2,SE3,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE3,euRBDCode,2021-12-02,None,None,None,None,...,swe,2009-12-22,None,riverBasinDistrict,nationalRiverBasinDistrict,None,None,None,None,63d40b39-4403-9c6e-2e06-18b767104d1b
3,SE4,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE4,euRBDCode,2021-11-30,None,None,None,None,...,swe,2009-12-22,None,riverBasinDistrict,nationalRiverBasinDistrict,None,None,None,None,bd7f5a8e-4f99-24b2-947e-1005e6218028
4,SE5_INT,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE5_INT,euRBDCode,2021-12-01,None,"SE5,SE5101",euRBDCode,None,...,swe,2022-09-05,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,ab29a96c-3079-23b8-40f6-40f2b7083ac3


## 5. Export to SQLite and GeoPackage

`record_id` is a deterministic UUID derived from the business key of each row (`meta.stable_record_id`). It is not imported into Reportnet 3, but the QCs report it, so it has to stay identical between the export and the QC run.

Export statements are `COPY` / `CREATE TABLE` statements and therefore cannot be stored as views; they are persisted as text in `meta.scripts` and executed from there.

In [36]:
output_dir = ROOT / "output" / countries_selection.value
sqlite_path = output_dir / "RiverBasinDistrictsAndCompetentAuthorities.sqlite"
geopackage_path = output_dir / "RiverBasinDistrict.gpkg"

wq.export_prefill(con, sqlite_path, geopackage_path)

Did you mean "SPATIAL_RiverBasinDistrict"?

LINE 17:     FROM prefill.SPATIAL_ProtectedArea
                  ^
Did you mean "CompetentAuthority"?
Did you mean "CompetentAuthority"?
Did you mean "CompetentAuthority"?
Did you mean "CompetentAuthority"?
Did you mean "CompetentAuthority"?


['SPATIAL_RiverBasinDistrict',
 'CompetentAuthority',
 'RiverBasinDistrictCompetentAuthority']

In [37]:
con.sql("SELECT name, target, sql_text FROM meta.scripts WHERE kind = 'export'").df()

,name,target,sql_text
0,CompetentAuthority,sqlite,-- export: CompetentAuthority -> SQLite\n-- Re...
1,derogation,sqlite,-- export: derogation -> SQLite\n-- Requires a...
2,exceedance,sqlite,-- export: exceedance -> SQLite\n-- Requires a...
3,monitoringresult,sqlite,-- export: monitoringresult -> SQLite\n-- Requ...
4,parameter,sqlite,-- export: parameter -> SQLite\n-- Requires an...
5,qualityandmonitoring,sqlite,-- export: qualityandmonitoring -> SQLite\n-- ...
6,RiverBasinDistrictCompetentAuthority,sqlite,-- export: RiverBasinDistrictCompetentAuthorit...
7,SPATIAL_ProtectedArea,geopackage,-- export: SPATIAL_ProtectedArea -> OGC GeoPac...
8,SPATIAL_RiverBasinDistrict,geopackage,-- export: SPATIAL_RiverBasinDistrict -> OGC G...


## 6. Review the exported spatial data

In [38]:
rbd_gdf = gpd.read_file(geopackage_path, layer="RiverBasinDistrict").to_crs(epsg=4326)

m = Map(
    center=(54.5260, 15.2551),
    zoom=4,
    basemap=basemaps.OpenStreetMap.Mapnik,
    scroll_wheel_zoom=True,
    layout={"height": "600px"},
)
m.add(LayersControl())
m.add(
    GeoData(
        geo_dataframe=rbd_gdf,
        name="RiverBasinDistrict",
        style={"color": "blue", "fillColor": "blue", "opacity": 0.5, "weight": 1.9, "dashArray": "5, 5"},
        hover_style={"fillColor": "red", "fillOpacity": 0.5},
    )
)
m

Map(center=[54.526, 15.2551], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [40]:
con.close()